In [1]:
import ipywidgets as widgets
from ipywidgets import interact
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go


from ift6758.data_acquisition import (
    load_game,
    draw_rink
)

saisons = [20162017, 20172018, 20182019, 20192020, 20202021, 20212022, 20222023, 20232024]
types_saison = {'Saison régulière': 'regular', 'Playoffs': 'playoffs'}

@interact(
    #Widget pour selectionner la saison
    season_id=widgets.Dropdown(options=saisons, value=saisons[0], description="Saison :"),
    #Widget pour selectionner le type de la saison
    season_type=widgets.Dropdown(options=types_saison, value="regular", description="Type :")
)
def interactive_game_id(season_id, season_type):
    #Récupère le dossier correspond à la saison choisis
    dossier = Path("../data/raw") / str(season_id) / season_type 

    #Récupère tous les matchs dans le dossier
    matchs = sorted([f.stem for f in dossier.glob("*.json")])

    #Widget pour selectionner le match
    @interact(game_id=widgets.SelectionSlider(options=matchs, value=matchs[0], description="ID du Match :", continuous_update=False))
    def interactive_event_id(game_id):

        #Récupère le dataframe du match
        df_game = load_game(season_id, season_type, int(game_id))

        #Récupère toutes les données nécessaires
        period = df_game['periodDescriptor.periodType'].iloc[0]
        homeTeam = df_game['homeTeam.commonName.default'].iloc[0]
        homeAbbrev = df_game['homeTeam.abbrev'].iloc[0]
        homeScore = df_game['homeTeam.score'].iloc[0]
        homeSog = df_game['homeTeam.sog'].iloc[0]
        awayTeam = df_game['awayTeam.commonName.default'].iloc[0]
        awayAbbrev = df_game['awayTeam.abbrev'].iloc[0]
        awayScore = df_game['awayTeam.score'].iloc[0]
        awaySog = df_game['awayTeam.sog'].iloc[0]

        #Affichage des données
        print(df_game['startTimeUTC'].iloc[0])
        print("Game ID :", game_id,"; ", homeAbbrev , "(home) vs ", awayAbbrev, "(away)")

        tableau = {
            'Home' : [homeTeam, homeAbbrev, homeScore, homeSog],
            'Away' : [awayTeam, awayAbbrev, awayScore, awaySog]
        }

        indices = ["Team :", "Abbrev :", "Score :", "SOG: "]

        data_game = pd.DataFrame(tableau,indices)
        print(period)
        display(data_game)

        #Récupère tous les événements
        events_list = df_game['plays'].iloc[0]
        nb_events = len(events_list)
        
        #Widget pour selectionner l'événement
        @interact(event_id=widgets.IntSlider(min=0,max=nb_events - 1, step=1, value=0, description="Événement numéro :", continuous_update=False))
        def info_event(event_id):
            #Récupère tous les infos nécessaire dans l'événement
            currentEvent = events_list[event_id]
            description = currentEvent.get('typeDescKey','')
            details = currentEvent.get('details') or {}
            xCoord = details.get('xCoord',0)
            yCoord = details.get('yCoord',0)

            #Affichage du rink avec les coordonnées de l'événement
            fig = go.Figure()
            fig.add_scatter(x=[xCoord], y=[yCoord], mode="markers", marker_size=18)
            fig.update_layout(title=f"Event : {description}")
            rink = draw_rink(fig)

            if rink is None:
                return

            rink.show()
            
            #Affichage des infos de l'événement
            display(currentEvent)



ModuleNotFoundError: No module named 'plotly'